# Stage 09 — Final econometric validation and robustness audit

This notebook audits the established Brent/WTI data, Stage 03–07 fits, forecasts and bootstrap calculations. It preserves every earlier file. New tables use the `09_` prefix. All fit sensitivities are developed on 2000–2015 training data and judged on 2016–2019 validation data; the previously examined 2020–2026 period is not used to select an alternative.

Return units are percentage points. The irregular-gap experiment divides each observed return by the square root of the number of intervening weekdays (ordinary Friday-to-Monday intervals have exposure one), fits the original observation-step models, and rescales predictions to raw-return units. This is a diffusion-style sensitivity, not a continuous-time or fully gap-aware Markov/GARCH likelihood. It retains all validation dates and leaves the original sample intact.

The same-day VIX experiment constructs the latest strictly earlier *observed* VIX quote with an as-of join, recovering 85 Brent returns. Results on the original 999 validation dates are the matched-date comparison. The seven-calendar-day GPR experiment is a publication-delay proxy using the revised historical index; it is not point-in-time data.

## Audit methods and uncertainty

The code independently reconstructs saved QLIKE, Gaussian and standardized Student-t densities, Markov mixture moments, filtered-state recursions, GARCH variance recursions and paired block-bootstrap intervals. The M4/M5 likelihood-ratio check fits on identical dates and observation equations. Its nominal chi-square p-value is labelled exploratory. A conditional parametric bootstrap simulates under M4 and re-estimates both models from two starts per replication. A separate M5 parametric bootstrap re-estimates transition contrasts. Optimizer failures are recorded and excluded from successful-replication distributions, never counted as null-tail evidence. These parametric bootstraps condition on revised observed predictor paths and inherit the fitted model assumptions.

In [1]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'code'))
from stage09_final_audit import run
audit = run()
print('Stage 09 computations and assertions completed.')

Stage 09 computations and assertions completed.


## Principal numerical outputs

The complete attempt ledgers, per-date forecasts, exposure ledger, software manifest, GPR source assessment and input hashes are saved under `results/tables/09_*`. This display is a compact index to those machine-readable results.

In [2]:
from IPython.display import display
for label in ('original_lr', 'original_effects', 'parametric_bootstrap', 'sensitivity_performance'):
    print('\n' + label)
    display(audit[label])
print('\nForecast verification')
display(audit['forecast_checks'])
print('\nOriginal paired bootstrap verification')
display(audit['paired_bootstrap'])


original_lr


,n,ll_m4,ll_m5,parameters_m4,parameters_m5,lr,nominal_chi2_df,nominal_chi2_p,inferential_status
0,3699,-8029.997107,-8025.590622,8,10,8.81297,2,0.012198,nominal only; parametric bootstrap reported se...



original_effects


,model,predictor,transition,p10_value,p90_value,probability_p10,probability_p90,difference,delta_se,ci95_low,ci95_high,saved_difference,saved_ci95_low,saved_ci95_high,absolute_difference_discrepancy,max_ci_bound_discrepancy
0,M5,gpr,low_to_high,52.714404,164.303809,0.016256,0.051954,0.035698,0.012753,0.010702,0.060694,0.03565,0.010656,0.060645,0.000048,0.000049
1,M5,vix,high_to_high,12.390000,31.178000,0.782661,0.975802,0.193140,0.085777,0.025019,0.361262,0.19298,0.025049,0.360910,0.000161,0.000352



parametric_bootstrap


,experiment,attempted,successful,failures,observed,bootstrap_p,monte_carlo_se,bootstrap_ci95_low,bootstrap_ci95_high,tail_count
0,lr_under_m4,120,120,0,8.812970,0.008264,0.00823,NaN,NaN,0.0
1,gpr_low_high,120,120,0,0.035698,NaN,NaN,0.016849,0.076690,NaN
2,vix_high_high,120,120,0,0.193140,NaN,NaN,0.106737,0.349072,NaN


... (2 outputs left out)


Forecast verification


,period,model,n,max_density_absolute_error,qlike_recomputed,target_return_leakage_check,scaling_check,matched_dates
0,validation,B0,999,4.410872e-11,2.564722,passed: prior filtered posterior predicts next...,passed,passed
1,validation,B1,999,1.410960e-11,2.446274,passed: prior filtered posterior predicts next...,passed,passed
2,validation,B1-t,999,7.424283e-12,2.448625,passed: prior filtered posterior predicts next...,passed,passed
3,validation,M1,999,6.354917e-12,2.442018,passed: prior filtered posterior predicts next...,passed,passed
4,validation,M3,999,6.279421e-12,2.447824,passed: prior filtered posterior predicts next...,passed,passed
5,validation,M4,999,5.563550e-12,2.442049,passed: prior filtered posterior predicts next...,passed,passed
6,validation,M5,999,6.388667e-12,2.445023,passed: prior filtered posterior predicts next...,passed,passed
7,test,B0,1712,4.653771e-10,4.225849,passed: prior filtered posterior predicts next...,passed,passed
8,test,B1,1712,1.359766e-10,2.925164,passed: prior filtered posterior predicts next...,passed,passed
9,test,B1-t,1712,5.268141e-11,2.944249,passed: prior filtered posterior predicts next...,passed,passed



Original paired bootstrap verification


,period,left,right,n,mean_difference,basic_ci95_low,basic_ci95_high,original_absolute_tail_p,equal_tail_p,interval_excludes_zero
0,validation,M1,M3,999,-0.005806,-0.014579,0.004049,0.198901,0.202899,False
1,validation,M1,M4,999,-0.000031,-0.018838,0.021407,0.996502,0.988506,False
2,validation,M1,M5,999,-0.003005,-0.020841,0.016882,0.763618,0.758621,False
3,validation,M4,M5,999,-0.002974,-0.008756,0.002866,0.307346,0.316842,False
4,validation,B1,M5,999,0.001251,-0.033153,0.037703,0.940530,0.936532,False
5,validation,B1-t,M5,999,0.003602,-0.032024,0.040373,0.849575,0.837581,False
6,test,M1,M3,1712,0.045574,-0.036339,0.097987,0.180410,0.235882,False
7,test,M1,M4,1712,0.087115,-0.036938,0.171938,0.098451,0.155922,False
8,test,M1,M5,1712,0.087905,-0.047299,0.180397,0.122439,0.175912,False
9,test,M4,M5,1712,0.000790,-0.014969,0.013094,0.905047,0.833583,False


## Interpretation rules

In-sample transition effects are descriptive associations, not causal effects. The official GPR archive does not establish a contemporaneous 2000–2019 information set. The historical GPR forecast exercise is therefore pseudo-out-of-sample. The seven-day proxy still uses revised values. Conditional bootstrap intervals assume a correct fitted Markov observation/transition specification, while paired block bootstrap intervals describe uncertainty in observed validation loss differences. Multiple model comparisons and latent-regime nonregularity prevent a small nominal p-value alone from establishing robust predictive value. The separate `results/09_final_audit_report.md` states the final verdict and thesis text implications.